# Q3 – LLM-Based ESG Message Triage

## Objective
Evaluate whether a Large Language Model can classify ESG operational messages, assess urgency, identify follow-up requirements, and recommend appropriate escalation pathways.

## Methodology
1. Prepare three synthetic ESG messages.
2. Design and evaluate an improved LLM prompt.
3. Generate structured JSON outputs.
4. Compare LLM classifications against a rule-based baseline.




## Phase 1 – Prompt Engineering and LLM Evaluation

### 1.1 Critical Evaluation of the Original Prompt
The original prompt requests eight structured ESG triage fields in JSON format. However, it provides limited classification guidance, which may produce inconsistent results across different messages.

**Identified limitations:**
1. **Undefined urgency criteria:** No clear distinction between LOW, MEDIUM, HIGH, and CRITICAL incidents.
2. **Ambiguous issue categories:** The model may generate inconsistent category labels.
3. **Unclear escalation rules:** The prompt does not specify how safety risks or accessibility barriers should be prioritised.
4. **Hallucination risk:** The model may infer unverified facts or assign inappropriate teams.
5. **Limited data protection:** No explicit instruction addresses personal information or sensitive allegations.

**Proposed improvement:** Introduce standardised labels, explicit decision rules, evidence-based classifications, privacy safeguards, and strict JSON output requirements.

In [1]:

import pandas as pd

# Synthetic ESG operational messages
test_messages = [
    {
        "id": "M01",
        "esg_dimension": "Environmental",
        "message": (
            "The air conditioning has been running overnight "
            "in an empty office on Level 3 for the past week. "
            "Could the facilities team check the schedule?"
        )
    },
    {
        "id": "M02",
        "esg_dimension": "Social",
        "message": (
            "The wheelchair-accessible entrance to Building B "
            "has been blocked by construction materials for two days. "
            "Staff using wheelchairs cannot access the building safely."
        )
    },
    {
        "id": "M03",
        "esg_dimension": "Governance / Environmental",
        "message": (
            "I heard that one of our suppliers may be disposing "
            "of hazardous waste improperly, but I haven't seen "
            "any evidence. I'm concerned that this could breach "
            "our sustainability policy."
        )
    }
]

df_messages = pd.DataFrame(test_messages)
display(df_messages)


,id,esg_dimension,message
0,M01,Environmental,The air conditioning has been running overnigh...
1,M02,Social,The wheelchair-accessible entrance to Building...
2,M03,Governance / Environmental,I heard that one of our suppliers may be dispo...


### 1.2 Revised Prompt Design

The revised prompt introduces standardised issue categories, explicit urgency thresholds, predefined team-routing rules, and safeguards against unsupported assumptions.

**Classification rules:**

- **LOW:** Routine issues with no immediate safety or accessibility impact.
- **MEDIUM:** Repeated operational problems requiring investigation.
- **HIGH:** Significant accessibility barriers, potential safety hazards, or serious compliance concerns.
- **CRITICAL:** Immediate threats to life or serious injury requiring emergency intervention.

The prompt distinguishes confirmed incidents from unverified allegations, requires a consistent JSON schema, and prohibits unsupported claims. Sensitive or ambiguous incidents should be referred for human review rather than automatically treated as verified facts.

These improvements aim to increase consistency, reduce hallucinations, and support safer ESG triage.

In [2]:

REVISED_PROMPT = """
You are an ESG operational message triage assistant.

TASK:
Analyse the employee message and return one valid JSON object
containing exactly the eight required fields.

ISSUE CATEGORIES:
- ENERGY_WASTE
- WATER_WASTE
- WASTE_MANAGEMENT
- ACCESSIBILITY
- SUPPLIER_COMPLIANCE
- OTHER

URGENCY:
- LOW: Routine issue with no immediate safety impact.
- MEDIUM: Repeated operational issue requiring investigation.
- HIGH: Significant accessibility barrier, potential safety
  hazard, or serious compliance concern.
- CRITICAL: Explicit immediate threat to life or serious injury.

SENTIMENT:
POSITIVE, NEUTRAL, or NEGATIVE.

FOLLOW-UP:
Y if the issue needs investigation or action; otherwise N.

TEAM ROUTING:
- ENERGY_WASTE: Facilities / Energy Management
- WATER_WASTE: Facilities Team
- WASTE_MANAGEMENT: Facilities / Waste Management
- ACCESSIBILITY: Accessibility / Facilities Team
- SUPPLIER_COMPLIANCE: Compliance / Procurement Team
- OTHER: Human Review

SAFETY AND GOVERNANCE RULES:
1. Base all classifications on the message only.
2. Do not invent facts, identities, evidence, or outcomes.
3. Distinguish allegations from verified incidents.
4. Escalate significant accessibility and safety risks.
5. Refer uncertain or sensitive concerns for human review.
6. Do not repeat personal identifying details unnecessarily.
7. Provide a concise escalation reason and summary.
8. Return JSON only, without markdown or explanations.

REQUIRED JSON FORMAT:
{
  "issue_category": "",
  "urgency": "",
  "sentiment": "",
  "followup_required": "",
  "recommended_team": "",
  "escalation_reason": "",
  "data_sensitivity_risk": "",
  "brief_summary": ""
}

EMPLOYEE MESSAGE:
{message}
"""
print("Revised prompt ready.")


Revised prompt ready.


### 1.3 OpenAI GPT-4o API Integration

OpenAI GPT-4o was selected as the primary LLM for ESG message triage, using an API key provided for coursework.

The model processes three synthetic ESG messages using the revised prompt and generates structured JSON outputs. API credentials are securely accessed through Google Colab Secrets.

The resulting classifications will be validated and compared against a rule-based baseline to assess consistency, routing accuracy, and potential errors.

In [3]:

!pip -q install -U openai

from openai import OpenAI
from google.colab import userdata
import json

# Read the API key from Colab Secrets
api_key = userdata.get("OPENAI_API_KEY")

# Connect to OpenAI
client = OpenAI(api_key=api_key)

MODEL_NAME = "gpt-4o"

print("OpenAI client initialized successfully.")
print("Selected model:", MODEL_NAME)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 17.4 MB/s eta 0:00:00
OpenAI client initialized successfully.
Selected model: gpt-4o


### 1.4 First LLM Inference – ESG Message M01

**Purpose:** Test the GPT-4o API using the first synthetic ESG message (M01) to evaluate its ability to classify an operational issue and generate structured JSON output.

The code retrieves M01 from the dataset, inserts it into the revised prompt, and sends it to GPT-4o with a low-temperature setting for more consistent responses. The model's response is then parsed and displayed as JSON containing the eight required ESG triage fields.

This initial test verifies the API execution and provides the first classification output for subsequent validation and comparison with the rule-based baseline.

In [4]:

# Select the first ESG message
message_m01 = df_messages.loc[
    df_messages["id"] == "M01", "message"
].iloc[0]

# Build the prompt
prompt_m01 = REVISED_PROMPT.replace(
    "{message}", message_m01
)

# Call GPT-4o API
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an ESG message triage assistant. "
                "Follow the provided classification rules "
                "and return valid JSON only."
            )
        },
        {
            "role": "user",
            "content": prompt_m01
        }
    ],
    temperature=0,
    response_format={"type": "json_object"}
)

# Parse JSON
result_m01 = json.loads(
    response.choices[0].message.content
)

print("MODEL:", response.model)
print("MESSAGE ID: M01")
print(json.dumps(result_m01, indent=2))


MODEL: gpt-4o-2024-08-06
MESSAGE ID: M01
{
  "issue_category": "ENERGY_WASTE",
  "urgency": "MEDIUM",
  "sentiment": "NEUTRAL",
  "followup_required": "Y",
  "recommended_team": "Facilities / Energy Management",
  "escalation_reason": "Repeated operational issue requiring investigation.",
  "data_sensitivity_risk": "Low",
  "brief_summary": "Air conditioning running overnight in empty office on Level 3 for a week."
}


**M01 – Result Interpretation**

GPT-4o successfully classified the message as ENERGY_WASTE with MEDIUM urgency, reflecting repeated unnecessary energy consumption over one week. The recommended Facilities / Energy Management team was consistent with the predefined routing rules.

All eight required JSON fields were returned, and no unsupported factual claims were identified in the response. This initial result demonstrates appropriate classification for a straightforward environmental scenario, although further testing is required to assess reliability across different ESG categories.

### 1.5 LLM Inference – Remaining ESG Scenarios

**Purpose:** Evaluate GPT-4o on two additional ESG messages involving accessibility barriers and an unverified supplier compliance concern.

The same revised prompt and model configuration are applied to both messages. The outputs are stored for JSON validation and subsequent comparison against the rule-based baseline.

In [5]:

# Preserve the existing M01 output
llm_results = {"M01": result_m01}

# Run M02 and M03 using the same model and prompt
for message_id in ["M02", "M03"]:
    message = df_messages.loc[
        df_messages["id"] == message_id, "message"
    ].iloc[0]

    prompt = REVISED_PROMPT.replace("{message}", message)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an ESG message triage assistant. "
                    "Follow the provided classification rules "
                    "and return valid JSON only."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    llm_results[message_id] = json.loads(
        response.choices[0].message.content
    )

    print(f"\n{'=' * 45}")
    print("MESSAGE ID:", message_id)
    print("MODEL:", response.model)
    print(json.dumps(llm_results[message_id], indent=2))



MESSAGE ID: M02
MODEL: gpt-4o-2024-08-06
{
  "issue_category": "ACCESSIBILITY",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Accessibility / Facilities Team",
  "escalation_reason": "Significant accessibility barrier preventing safe access.",
  "data_sensitivity_risk": "Low",
  "brief_summary": "Wheelchair-accessible entrance to Building B blocked by construction materials, hindering safe access."
}

MESSAGE ID: M03
MODEL: gpt-4o-2024-08-06
{
  "issue_category": "SUPPLIER_COMPLIANCE",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Compliance / Procurement Team",
  "escalation_reason": "Potential serious compliance concern regarding hazardous waste disposal.",
  "data_sensitivity_risk": "Medium due to unverified allegations.",
  "brief_summary": "Allegation of improper hazardous waste disposal by a supplier, potentially breaching sustainability policy."
}


**M02–M03: Initial Result Interpretation**

GPT-4o correctly identified the accessibility barrier (M02) and supplier compliance concern (M03), assigning HIGH urgency and recommending relevant support teams for both cases.

The model preserved the unverified nature of the supplier allegation rather than treating it as a confirmed breach. However, the data_sensitivity_risk field was inconsistent across responses, indicating that the revised prompt requires stricter output constraints.

These findings demonstrate promising contextual classification but also highlight the need for schema validation, explicit sensitivity labels, and human review of sensitive or high-priority cases.

### 1.6 JSON Output Validation

**Purpose:** Validate the structure and consistency of GPT-4o outputs across all three ESG messages.

The validation checks the eight required fields, permitted category and urgency labels, sentiment, follow-up status, and data sensitivity classification. Any violations are recorded without modifying the original model outputs, allowing transparent evaluation of prompt reliability.

In [6]:
import pandas as pd

required_fields = {
    "issue_category",
    "urgency",
    "sentiment",
    "followup_required",
    "recommended_team",
    "escalation_reason",
    "data_sensitivity_risk",
    "brief_summary"
}

allowed_values = {
    "issue_category": {
        "ENERGY_WASTE", "WATER_WASTE", "WASTE_MANAGEMENT",
        "ACCESSIBILITY", "SUPPLIER_COMPLIANCE", "OTHER"
    },
    "urgency": {"LOW", "MEDIUM", "HIGH", "CRITICAL"},
    "sentiment": {"POSITIVE", "NEUTRAL", "NEGATIVE"},
    "followup_required": {"Y", "N"},
    "data_sensitivity_risk": {"LOW", "MEDIUM", "HIGH"}
}

validation_results = []

for message_id, result in llm_results.items():
    errors = []

    missing = required_fields - set(result)
    extra = set(result) - required_fields

    if missing:
        errors.append(f"Missing: {sorted(missing)}")
    if extra:
        errors.append(f"Unexpected: {sorted(extra)}")

    for field, allowed in allowed_values.items():
        if field in result and result[field] not in allowed:
            errors.append(
                f"{field}: invalid value '{result[field]}'"
            )

    for field in required_fields:
        if field in result and (
            not isinstance(result[field], str)
            or not result[field].strip()
        ):
            errors.append(f"{field}: empty or non-string value")

    validation_results.append({
        "Message ID": message_id,
        "Schema Status": "PASS" if not errors else "FAIL",
        "Validation Issues": "; ".join(errors) if errors else "None"
    })

df_validation = pd.DataFrame(validation_results)
display(df_validation)


,Message ID,Schema Status,Validation Issues
0,M01,FAIL,data_sensitivity_risk: invalid value 'Low'
1,M02,FAIL,data_sensitivity_risk: invalid value 'Low'
2,M03,FAIL,data_sensitivity_risk: invalid value 'Medium d...


**Validation Results Interpretation**

All three GPT-4o responses contained the eight required JSON fields but failed strict schema validation due to inconsistent data_sensitivity_risk values.

The results reveal a limitation in the revised prompt: while it defines categories, urgency and routing rules, it does not explicitly constrain data sensitivity labels.

This demonstrates that syntactically valid JSON does not guarantee compliance with business requirements. Stricter output constraints and automated validation are therefore needed before operational deployment.

## Phase 2 – Rule-Based Baseline and Comparison

### 2.1 Baseline Design

**Purpose:** Develop a transparent rule-based classifier to provide a comparison benchmark for GPT-4o ESG message triage.

The baseline uses predefined keywords and decision rules to identify issue categories, assign urgency levels, and recommend responsible teams. The same three synthetic ESG messages are used to ensure a consistent comparison.

Unlike GPT-4o, the baseline does not interpret complex contextual meaning. This allows the experiment to assess whether LLM-based classification provides additional value over simpler deterministic rules.

In [7]:
CATEGORY_RULES = {
    "ACCESSIBILITY": [
        "wheelchair", "accessible entrance", "accessibility"
    ],
    "SUPPLIER_COMPLIANCE": [
        "supplier", "sustainability policy", "compliance"
    ],
    "ENERGY_WASTE": [
        "air conditioning", "electricity", "energy"
    ],
    "WATER_WASTE": [
        "water leak", "leaking tap", "water waste"
    ],
    "WASTE_MANAGEMENT": [
        "recycling bin", "waste disposal", "contamination"
    ]
}

TEAM_MAPPING = {
    "ACCESSIBILITY": "Accessibility / Facilities Team",
    "SUPPLIER_COMPLIANCE": "Compliance / Procurement Team",
    "ENERGY_WASTE": "Facilities / Energy Management",
    "WATER_WASTE": "Facilities Team",
    "WASTE_MANAGEMENT": "Facilities / Waste Management",
    "OTHER": "Human Review"
}

def rule_based_triage(message):
    text = message.lower()

    # Category classification using keyword matching
    category = "OTHER"

    for label, keywords in CATEGORY_RULES.items():
        if any(keyword in text for keyword in keywords):
            category = label
            break

    # Basic urgency rules
    if any(x in text for x in [
        "immediate danger", "life-threatening", "serious injury"
    ]):
        urgency = "CRITICAL"
    elif any(x in text for x in [
        "wheelchair", "hazardous", "unsafe"
    ]):
        urgency = "HIGH"
    elif any(x in text for x in [
        "past week", "two days", "repeated", "overnight"
    ]):
        urgency = "MEDIUM"
    else:
        urgency = "LOW"

    # Simple sentiment classification
    if any(x in text for x in [
        "concerned", "cannot", "blocked", "unsafe"
    ]):
        sentiment = "NEGATIVE"
    else:
        sentiment = "NEUTRAL"

    return {
        "issue_category": category,
        "urgency": urgency,
        "sentiment": sentiment,
        "followup_required": "Y",
        "recommended_team": TEAM_MAPPING[category]
    }

print("Rule-based ESG classifier ready.")


Rule-based ESG classifier ready.


In [8]:

# Run baseline on the same ESG test messages

baseline_results = {}

for _, row in df_messages.iterrows():
    message_id = row["id"]

    baseline_results[message_id] = rule_based_triage(
        row["message"]
    )

df_baseline = pd.DataFrame.from_dict(
    baseline_results,
    orient="index"
)

df_baseline.index.name = "Message ID"

display(df_baseline)


,issue_category,urgency,sentiment,followup_required,recommended_team
Message ID,,,,,
M01,ENERGY_WASTE,MEDIUM,NEUTRAL,Y,Facilities / Energy Management
M02,ACCESSIBILITY,HIGH,NEGATIVE,Y,Accessibility / Facilities Team
M03,SUPPLIER_COMPLIANCE,HIGH,NEGATIVE,Y,Compliance / Procurement Team


### 2.2 LLM vs Rule-Based Baseline Comparison

**Purpose:** Compare GPT-4o and the rule-based baseline on five common triage fields: issue category, urgency, sentiment, follow-up requirement, and recommended team.

The comparison measures field-level agreement between the two approaches and identifies classification differences. Agreement does not necessarily indicate accuracy, as both methods may produce the same incorrect prediction.



In [9]:
import pandas as pd

comparison_fields = [
    "issue_category",
    "urgency",
    "sentiment",
    "followup_required",
    "recommended_team"
]

comparison_rows = []

for message_id in ["M01", "M02", "M03"]:
    llm = llm_results[message_id]
    baseline = baseline_results[message_id]

    for field in comparison_fields:
        llm_value = llm.get(field)
        baseline_value = baseline.get(field)

        comparison_rows.append({
            "Message ID": message_id,
            "Field": field,
            "GPT-4o": llm_value,
            "Rule-Based": baseline_value,
            "Match": llm_value == baseline_value
        })

df_comparison = pd.DataFrame(comparison_rows)

display(df_comparison)


,Message ID,Field,GPT-4o,Rule-Based,Match
0,M01,issue_category,ENERGY_WASTE,ENERGY_WASTE,True
1,M01,urgency,MEDIUM,MEDIUM,True
2,M01,sentiment,NEUTRAL,NEUTRAL,True
3,M01,followup_required,Y,Y,True
4,M01,recommended_team,Facilities / Energy Management,Facilities / Energy Management,True
5,M02,issue_category,ACCESSIBILITY,ACCESSIBILITY,True
6,M02,urgency,HIGH,HIGH,True
7,M02,sentiment,NEGATIVE,NEGATIVE,True
8,M02,followup_required,Y,Y,True
9,M02,recommended_team,Accessibility / Facilities Team,Accessibility / Facilities Team,True


In [10]:

# Calculate agreement between GPT-4o and baseline

overall_agreement = df_comparison["Match"].mean() * 100

field_agreement = (
    df_comparison.groupby("Field")["Match"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

field_agreement["Agreement (%)"] = (
    field_agreement["mean"] * 100
).round(1)

field_agreement = field_agreement[
    ["Field", "sum", "count", "Agreement (%)"]
]

field_agreement.columns = [
    "Field", "Matches", "Total", "Agreement (%)"
]

print(f"Overall agreement: {overall_agreement:.1f}%")
display(field_agreement)


Overall agreement: 100.0%


,Field,Matches,Total,Agreement (%)
0,followup_required,3,3,100.0
1,issue_category,3,3,100.0
2,recommended_team,3,3,100.0
3,sentiment,3,3,100.0
4,urgency,3,3,100.0


**Comparison Results Interpretation**

GPT-4o and the rule-based baseline achieved 100% field-level agreement (15/15 comparisons) across the three synthetic ESG messages. Both approaches assigned identical issue categories, urgency levels, sentiment labels, follow-up decisions, and recommended teams.

However, agreement does not establish classification accuracy. The test messages contain relatively explicit keywords, making them suitable for deterministic rules. Furthermore, the baseline rules were developed around the same scenarios, limiting the independence of this comparison.

GPT-4o additionally generated contextual summaries and escalation reasons, but its sensitivity-risk outputs failed strict schema validation. More challenging unseen messages are therefore needed to examine differences in contextual understanding and error handling.

### 2.3 Error and Bias Analysis

**Purpose:** Evaluate the robustness of GPT-4o and the rule-based baseline using two additional unseen ESG messages.

The challenge tests focus on:
1. **Negation and resolved incidents:** Whether the classifiers recognise that an accessibility issue is no longer active.
2. **Alternative wording:** Whether the classifiers recognise an energy-waste issue without relying on predefined keywords.

The original prompt, model settings, and baseline rules remain unchanged. Differences will be reviewed against the messages' meaning rather than assuming that either classifier is automatically correct.

In [11]:
# Additional unseen challenge messages

challenge_messages = [
    {
        "id": "M04",
        "test_type": "Negation / Resolved Incident",
        "message": (
            "The wheelchair-accessible entrance was blocked "
            "yesterday, but the materials have now been removed "
            "and access is fully restored. No further action is needed."
        )
    },
    {
        "id": "M05",
        "test_type": "Alternative Wording",
        "message": (
            "The lights on Level 4 remain switched on throughout "
            "the weekend even though nobody is using the offices. "
            "Please investigate."
        )
    }
]

df_challenge = pd.DataFrame(challenge_messages)
display(df_challenge)


,id,test_type,message
0,M04,Negation / Resolved Incident,The wheelchair-accessible entrance was blocked...
1,M05,Alternative Wording,The lights on Level 4 remain switched on throu...


In [12]:

# Compare GPT-4o and the existing baseline on challenge cases

challenge_results = []

for _, row in df_challenge.iterrows():
    message_id = row["id"]
    message = row["message"]

    # Existing rule-based baseline
    baseline = rule_based_triage(message)

    # GPT-4o using the unchanged revised prompt
    prompt = REVISED_PROMPT.replace("{message}", message)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an ESG message triage assistant. "
                    "Follow the provided classification rules "
                    "and return valid JSON only."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    llm = json.loads(response.choices[0].message.content)

    challenge_results.append({
        "Message ID": message_id,
        "Test Type": row["test_type"],
        "GPT-4o Category": llm.get("issue_category"),
        "Baseline Category": baseline.get("issue_category"),
        "GPT-4o Urgency": llm.get("urgency"),
        "Baseline Urgency": baseline.get("urgency"),
        "GPT-4o Follow-up": llm.get("followup_required"),
        "Baseline Follow-up": baseline.get("followup_required")
    })

    print(f"\n--- {message_id}: GPT-4o Full JSON ---")
    print(json.dumps(llm, indent=2))

df_challenge_results = pd.DataFrame(challenge_results)

print("\n--- Challenge Comparison Summary ---")
display(df_challenge_results)



--- M04: GPT-4o Full JSON ---
{
  "issue_category": "ACCESSIBILITY",
  "urgency": "LOW",
  "sentiment": "POSITIVE",
  "followup_required": "N",
  "recommended_team": "Accessibility / Facilities Team",
  "escalation_reason": "",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Wheelchair-accessible entrance was temporarily blocked but is now cleared and accessible."
}

--- M05: GPT-4o Full JSON ---
{
  "issue_category": "ENERGY_WASTE",
  "urgency": "MEDIUM",
  "sentiment": "NEUTRAL",
  "followup_required": "Y",
  "recommended_team": "Facilities / Energy Management",
  "escalation_reason": "Repeated operational issue requiring investigation.",
  "data_sensitivity_risk": "Low",
  "brief_summary": "Lights on Level 4 are left on during weekends without use."
}

--- Challenge Comparison Summary ---


,Message ID,Test Type,GPT-4o Category,Baseline Category,GPT-4o Urgency,Baseline Urgency,GPT-4o Follow-up,Baseline Follow-up
0,M04,Negation / Resolved Incident,ACCESSIBILITY,ACCESSIBILITY,LOW,HIGH,N,Y
1,M05,Alternative Wording,ENERGY_WASTE,OTHER,MEDIUM,LOW,Y,Y


**Challenge Test Interpretation**

The challenge tests confirmed two weaknesses in the rule-based baseline. For M04, the baseline assigned HIGH urgency and required follow-up despite the incident being resolved, indicating a failure to interpret negation and incident status. GPT-4o correctly assigned LOW urgency and no follow-up.

For M05, the baseline classified unnecessary lighting as OTHER due to missing keywords, while GPT-4o correctly identified ENERGY_WASTE and assigned MEDIUM urgency.

These findings suggest that GPT-4o offers stronger contextual interpretation in the tested scenarios. However, the small synthetic sample limits generalisability, and the experiment does not establish systematic demographic fairness or bias performance.

## Phase 3 – Prompt Refinement and Production Readiness

### 3.1 Prompt Improvement

**Purpose:** Refine the original ESG triage prompt based on observed schema-validation failures and challenge-test findings.

The revised version introduces strict sensitivity labels, mandatory explanations, resolved-incident handling, and safeguards for unverified allegations.

The original GPT-4o outputs are preserved as experimental evidence. Prompt Version 2 will be evaluated separately to assess whether these refinements improve output consistency.

In [13]:

# Phase 4.1 – Improved ESG Triage Prompt (Version 2)

REVISED_PROMPT_V2 = REVISED_PROMPT.replace(
    "SAFETY AND GOVERNANCE RULES:",
    """DATA SENSITIVITY CLASSIFICATION:
- LOW: No identifiable personal or confidential information.
- MEDIUM: Potentially sensitive workplace allegations,
  accessibility concerns, or confidential operational details.
- HIGH: Explicit sensitive personal data, identifiable
  individuals, or highly confidential allegations.
- Return ONLY LOW, MEDIUM, or HIGH.

INCIDENT STATUS RULES:
- Distinguish active incidents from resolved incidents.
- If explicitly resolved with no further action needed,
  assign LOW urgency and followup_required = N unless
  an ongoing safety or compliance risk is stated.
- Do not infer that an unverified allegation is confirmed.
- For uncertain or sensitive concerns, recommend human review.

OUTPUT REQUIREMENTS:
- All eight fields must be present.
- All fields must contain a non-empty string.
- escalation_reason must explain why escalation is required,
  or state "No escalation required" when appropriate.
- Use only the permitted labels for categories, urgency,
  sentiment, follow-up, and data sensitivity.

SAFETY AND GOVERNANCE RULES:"""
)

print("Improved prompt V2 ready.")


Improved prompt V2 ready.


### 3.2 Prompt V2 Re-testing and Validation

**Purpose:** Evaluate whether the improved ESG triage prompt resolves the schema inconsistencies and output-quality issues identified during earlier experiments.

The same five synthetic ESG messages (M01–M05) are processed using GPT-4o with Prompt V2. The original model and temperature settings remain unchanged to support a controlled comparison.

The resulting JSON outputs are validated against predefined field and label requirements, then compared with the original Prompt V1 results. Particular attention is given to data sensitivity labels, escalation explanations, and the interpretation of resolved incidents.

In [14]:

import json
import pandas as pd

# Combine original and challenge datasets
df_all_messages = pd.concat(
    [
        df_messages[["id", "message"]],
        df_challenge[["id", "message"]]
    ],
    ignore_index=True
)

# Store Prompt V2 outputs separately from V1
llm_results_v2 = {}
experiment_log_v2 = []

for _, row in df_all_messages.iterrows():
    message_id = row["id"]
    message = row["message"]

    prompt = REVISED_PROMPT_V2.replace(
        "{message}", message
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an ESG message triage assistant. "
                    "Follow the classification and safety rules. "
                    "Return a valid JSON object only."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    result = json.loads(
        response.choices[0].message.content
    )

    llm_results_v2[message_id] = result

    experiment_log_v2.append({
        "message_id": message_id,
        "prompt_version": "V2",
        "model": response.model,
        "temperature": 0,
        "result": result
    })

    print(f"\n--- {message_id}: Prompt V2 ---")
    print(json.dumps(result, indent=2))

print("\nTotal messages processed:", len(llm_results_v2))



--- M01: Prompt V2 ---
{
  "issue_category": "ENERGY_WASTE",
  "urgency": "MEDIUM",
  "sentiment": "NEUTRAL",
  "followup_required": "Y",
  "recommended_team": "Facilities / Energy Management",
  "escalation_reason": "Repeated operational issue requiring investigation.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Air conditioning running overnight in empty office on Level 3 for a week."
}

--- M02: Prompt V2 ---
{
  "issue_category": "ACCESSIBILITY",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Accessibility / Facilities Team",
  "escalation_reason": "Significant accessibility barrier and potential safety hazard.",
  "data_sensitivity_risk": "MEDIUM",
  "brief_summary": "Wheelchair-accessible entrance to Building B blocked by construction materials, preventing safe access."
}

--- M03: Prompt V2 ---
{
  "issue_category": "SUPPLIER_COMPLIANCE",
  "urgency": "MEDIUM",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",


In [15]:

# Phase 4.2 – Strict validation of Prompt V2 outputs

required_fields = {
    "issue_category",
    "urgency",
    "sentiment",
    "followup_required",
    "recommended_team",
    "escalation_reason",
    "data_sensitivity_risk",
    "brief_summary"
}

allowed_values = {
    "issue_category": {
        "ENERGY_WASTE", "WATER_WASTE",
        "WASTE_MANAGEMENT", "ACCESSIBILITY",
        "SUPPLIER_COMPLIANCE", "OTHER"
    },
    "urgency": {"LOW", "MEDIUM", "HIGH", "CRITICAL"},
    "sentiment": {"POSITIVE", "NEUTRAL", "NEGATIVE"},
    "followup_required": {"Y", "N"},
    "data_sensitivity_risk": {"LOW", "MEDIUM", "HIGH"}
}

validation_v2 = []

for message_id, result in llm_results_v2.items():
    errors = []

    # Check exact required fields
    missing = required_fields - set(result)
    extra = set(result) - required_fields

    if missing:
        errors.append(f"Missing fields: {sorted(missing)}")

    if extra:
        errors.append(f"Unexpected fields: {sorted(extra)}")

    # Check permitted labels
    for field, values in allowed_values.items():
        if field in result and result[field] not in values:
            errors.append(
                f"{field}: invalid value '{result[field]}'"
            )

    # Check for empty or non-string values
    for field in required_fields:
        if field in result:
            value = result[field]
            if not isinstance(value, str) or not value.strip():
                errors.append(f"{field}: empty or invalid")

    validation_v2.append({
        "Message ID": message_id,
        "Schema Status": "PASS" if not errors else "FAIL",
        "Validation Issues": "; ".join(errors) if errors else "None"
    })

df_validation_v2 = pd.DataFrame(validation_v2)

display(df_validation_v2)

print(
    "Schema pass rate:",
    f"{(df_validation_v2['Schema Status'] == 'PASS').mean() * 100:.1f}%"
)


,Message ID,Schema Status,Validation Issues
0,M01,PASS,None
1,M02,PASS,None
2,M03,PASS,None
3,M04,PASS,None
4,M05,PASS,None


Schema pass rate: 100.0%


In [16]:

# Compare sensitivity outputs between V1 and V2

comparison_v1_v2 = []

for message_id in ["M01", "M02", "M03"]:
    v1 = llm_results[message_id]
    v2 = llm_results_v2[message_id]

    comparison_v1_v2.append({
        "Message ID": message_id,
        "V1 Sensitivity": v1.get("data_sensitivity_risk"),
        "V2 Sensitivity": v2.get("data_sensitivity_risk"),
        "V1 Valid": v1.get("data_sensitivity_risk") in {
            "LOW", "MEDIUM", "HIGH"
        },
        "V2 Valid": v2.get("data_sensitivity_risk") in {
            "LOW", "MEDIUM", "HIGH"
        }
    })

df_prompt_comparison = pd.DataFrame(comparison_v1_v2)

display(df_prompt_comparison)


,Message ID,V1 Sensitivity,V2 Sensitivity,V1 Valid,V2 Valid
0,M01,Low,LOW,False,True
1,M02,Low,MEDIUM,False,True
2,M03,Medium due to unverified allegations.,MEDIUM,False,True


**Prompt V2 Evaluation**

Prompt refinement improved strict schema compliance from 0/3 valid outputs in V1 to 3/3 in V2 for the original test messages. V2 also passed validation on both additional challenge messages, achieving 5/5 valid outputs overall.

The improvements standardised data sensitivity labels, introduced explicit explanations for resolved incidents, and preserved contextual classification of alternative wording.

However, M03 urgency changed from HIGH to MEDIUM, demonstrating that prompt modifications can also influence risk assessment. Therefore, schema compliance alone does not establish classification accuracy or operational safety. Further testing against independently defined reference labels is required.

### 3.3 Production Readiness Assessment

**Purpose:** Assess whether the refined ESG triage approach is suitable for operational deployment based on the observed experimental results.

**Findings and recommendations:**

- **Classification reliability:** GPT-4o handled contextual and resolved incidents better than the rule-based baseline in the challenge tests. However, five synthetic messages are insufficient to establish general reliability.
- **Structured outputs:** Prompt V2 improved schema compliance to 100% across five test messages. Production systems should additionally enforce schema validation at the API and application levels.
- **Escalation risk:** Incorrect urgency classifications could delay critical incidents or cause unnecessary escalation. HIGH and CRITICAL cases should follow predefined escalation policies with human oversight.
- **Data protection:** Employee messages may contain personal information, confidential allegations, or workplace details. Data minimisation, access control, retention policies, and appropriate redaction are required.
- **Human oversight:** Sensitive supplier allegations and uncertain classifications should be reviewed before operational decisions are taken.

**Recommendation:** The approach is suitable for a controlled pilot using synthetic or appropriately authorised data, but it is not production-ready. A larger independently labelled evaluation dataset, security controls, monitoring, and human review procedures are required before enterprise deployment.